In [ ]:
from office365.runtime.auth.client_credential import ClientCredential
from office365.sharepoint.client_context import ClientContext
from office365.sharepoint.files.file import File
import pandas as pd
import polars as pl
import io
from collections import namedtuple

In [ ]:
import json
import boto3
from botocore.exceptions import ClientError

def get_secret(SECRET_NAME, REGION_NAME, SECRET_KEY):

    secret_name = SECRET_NAME
    region_name = REGION_NAME

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    else:
        secret = get_secret_value_response['SecretString']
        decypted_api_keys = json.loads(secret)

        value =  decypted_api_keys[SECRET_KEY]
    
    return value

In [ ]:
CLIENT_ID = get_secret("dev/streamlit-regression-test/credentials", "us-east-1", "share_point_client_id")
CLIENT_SECRET = get_secret("dev/streamlit-regression-test/credentials", "us-east-1", "share_point_client_secret")
print(f"client-id: {CLIENT_ID}, client-sec: {CLIENT_SECRET}")

In [ ]:

sharepoint_url = 'https://share.tata.com/sites/MOT-EngineeringInfraTeams/'
client_credentials = ClientCredential(CLIENT_ID, CLIENT_SECRET)

In [ ]:
try:
    ctx = ClientContext(sharepoint_url).with_credentials(client_credentials)
except Exception as e:
    print(f"Error connecting to SharePoint: {str(e)}")


In [ ]:

sharepoint_folder = "General/Regression_files/Sales Navigator QA/Post-Prod Checks"
file_name = "test.xlsx"
file_path =  "/sites/MOT-EngineeringInfraTeams/Shared Documents/" + sharepoint_folder + "/" + file_name

try:
    file = ctx.web.get_file_by_server_relative_url(file_path)
    ctx.load(file)
    ctx.execute_query()
    response = File.open_binary(ctx, file.serverRelativeUrl)
    excel_io = io.BytesIO(response.content)      
except Exception as e:
    print(f"Error reading Excel file from SharePoint: {str(e)}")

df = pl.read_excel(excel_io, engine='openpyxl')


In [ ]:
print(df.columns)
print(df.rows())

In [ ]:
# Define NamedTuple
Row = namedtuple("Row", df.columns)

# Process each row safely
for row in map(Row._make, df.rows()):
    print(row.test_case, row.comments, row.process, row.expected_result, row.validation_sql)
    print('\n')